# NB04 — Transformer Causal (Décodeur-only)
> **Phase C & D** : Vaswani et al. 2017, Radford et al. 2019 (GPT-2), Xiong et al. 2020 (Pre-LN).


In [ ]:
# ==============================================================================
# Bootstrap : détection du répertoire racine (local ET Google Colab)
# ==============================================================================
import sys
from pathlib import Path

def _find_root():
    """Walk up from cwd looking for utils.py (also checks <p>/project1/utils.py)."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for p in candidates:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            # utils.py lives at repo root, not inside project1
            return p
    return None

ROOT_DIR = _find_root()

if ROOT_DIR is None:
    # ── Colab fallback: clone the repo ──────────────────────────────────────
    if Path("/content").exists():
        import subprocess
        clone_target = Path("/content/Recherche_AI")
        if not clone_target.exists():
            subprocess.run(
                ["git", "clone", "https://github.com/AhmadBelhaj4/Recherche_AI.git",
                 str(clone_target)],
                check=True,
            )
        ROOT_DIR = _find_root()

if ROOT_DIR is None:
    raise FileNotFoundError(
        "Could not locate utils.py. "
        "Set AI_PROJECT_DIR or run from inside the Recherche_AI repo."
    )

sys.path.insert(0, str(ROOT_DIR))
from utils import get_paths
PATHS = get_paths()
print(f"ROOT_DIR  : {ROOT_DIR}")
print(f"PATHS     : {PATHS}")


In [ ]:
# ==============================================================================
# Cellule 1 : Paramètres (Settings) et reproductibilité
# ==============================================================================
import sys
import math
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn

from utils import (set_seed, get_device, AminoAcidTokenizer,
                   SequenceDataset, collate_fn, make_loader, shift,
                   compute_nats_and_perplexity)
from models import CausalTransformerLM

SEED = 42
set_seed(SEED)
device = get_device()

BATCH_SIZE = 64
EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 6
DIM_FEEDFORWARD = 512
DROPOUT = 0.1
LR = 6e-4
WEIGHT_DECAY = 0.01
EPOCHS = 15
GRAD_CLIP = 1.0

print(f"Device: {device} | Seed: {SEED} | Layers: {NUM_LAYERS} | Heads: {NUM_HEADS}")


In [ ]:
# ==============================================================================
# Cellule 2 : Architecture Transformer Causal et Test Unitaire du Masque
# ==============================================================================
# CausalTransformerLM est défini dans models.py (importé dans la cellule 1)

tokenizer = AminoAcidTokenizer()
model = CausalTransformerLM(tokenizer.vocab_size, EMBED_DIM, NUM_HEADS, NUM_LAYERS, DIM_FEEDFORWARD).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Transformer instancié : {n_params:,} paramètres.")

# Test unitaire : Le futur ne doit pas affecter le passé
model.eval()
with torch.no_grad():
    x1 = torch.tensor([[1, 5, 8, 12]], device=device)
    x2 = torch.tensor([[1, 5, 8, 20]], device=device)  # 4ème token différent
    logits1 = model(x1)
    logits2 = model(x2)
    # Les logits aux positions 0, 1, 2 doivent être strictement identiques
    diff = (logits1[:, :3, :] - logits2[:, :3, :]).abs().max().item()
    print(f"Différence passée : {diff:.6e}")
    assert diff < 1e-5, "Violation de la causalité !"
print("Test unitaire causalité validé avec succès.")


In [ ]:
# ==============================================================================
# Cellule 3 : Entraînement Transformer (15 epochs) et sauvegarde
# ==============================================================================
train_path = PATHS["raw"].parent / "train.csv"
val_path = PATHS["raw"].parent / "val.csv"
if not train_path.exists():
    train_path = PATHS["raw"].parent / "toy" / "train.csv"
    val_path = PATHS["raw"].parent / "toy" / "val.csv"

train_loader = make_loader(
    pd.read_csv(train_path)["sequence"].tolist(), tokenizer,
    batch_size=BATCH_SIZE, shuffle=True,
)
val_loader = make_loader(
    pd.read_csv(val_path)["sequence"].tolist(), tokenizer,
    batch_size=BATCH_SIZE, shuffle=False,
)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id, reduction='sum')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

models_dir = PATHS["models"]
models_dir.mkdir(parents=True, exist_ok=True)
best_transformer_path = models_dir / "transformer_best.pt"
best_val_nats = float('inf')

print("Début de l'entraînement Transformer...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for batch in train_loader:
        inputs, targets = shift(batch)
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
        non_pad = (targets != tokenizer.pad_id).sum().item()
        (loss / max(non_pad, 1)).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()
        total_tokens += non_pad

    train_nats, train_ppl = compute_nats_and_perplexity(total_loss, total_tokens)
    model.eval()
    val_loss, val_tokens = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            inputs, targets = shift(batch)
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
            val_loss += loss.item()
            val_tokens += (targets != tokenizer.pad_id).sum().item()
    val_nats, val_ppl = compute_nats_and_perplexity(val_loss, val_tokens)

    if val_nats < best_val_nats:
        best_val_nats = val_nats
        torch.save(model.state_dict(), best_transformer_path)
        star = "★"
    else:
        star = ""

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train nats: {train_nats:.3f} | Val nats: {val_nats:.3f} (ppl: {val_ppl:.1f}) {star}")

print(f"Terminé. Meilleur modèle sauvegardé dans {best_transformer_path}")
